In [ ]:
import os
import numpy as np
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, ConvLSTM2D, Conv2D, TimeDistributed,
    Add, Activation, BatchNormalization, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, LearningRateScheduler, ReduceLROnPlateau
from tensorflow.keras.metrics import RootMeanSquaredError, MeanAbsolutePercentageError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt

# =====================================================
# CONFIGURACIÓN DE RUTAS
# =====================================================

d_original = '/home/fisica/bandas/banda13/ene_01_mes'
d_reducida = '/home/fisica/bandas/banda13/ene_01_reducida_mes'

if not os.path.exists(d_reducida):
    os.makedirs(d_reducida)

# =====================================================
# FUNCIONES AUXILIARES
# =====================================================

def resize_npy_images(d_original, d_reducida, new_size=(480, 480)):
    print(f"Redimensionando imágenes en {d_original}...")
    
    if not os.path.exists(d_reducida):
        os.makedirs(d_reducida)
    
    processed_count = 0
    for filename in os.listdir(d_original):
        if filename.lower().endswith('.npy'):
            img_path = os.path.join(d_original, filename)
            img_array = np.load(img_path)
            img_resized = np.array(Image.fromarray(img_array).resize(new_size, Image.LANCZOS))
            output_path = os.path.join(d_reducida, filename)
            np.save(output_path, img_resized)
            processed_count += 1
    
    print(f"Procesadas {processed_count} imágenes")
    return processed_count

def mostrar_resultados(X, y, y_pred, n=5, cmap='viridis',
                       titulo_general="Comparación de Resultados",
                       xlabel="Eje X (pixeles)", ylabel="Eje Y (pixeles)"):
    n = min(n, len(X))
    
    fig, axs = plt.subplots(n, 3, figsize=(12, 3 * n))
    fig.suptitle(titulo_general, fontsize=16, y=1.02)
    
    for i in range(n):
        axs[i, 0].imshow(X[i, ..., 0], cmap=cmap)
        axs[i, 0].set_title(f'Entrada t={i}')
        axs[i, 0].set_xlabel(xlabel)
        axs[i, 0].set_ylabel(ylabel)
        
        axs[i, 1].imshow(y[i, ..., 0], cmap=cmap)
        axs[i, 1].set_title(f'Real t+1={i+1}')
        axs[i, 1].set_xlabel(xlabel)
        axs[i, 1].set_ylabel(ylabel)
        
        axs[i, 2].imshow(y_pred[i, ..., 0], cmap=cmap)
        axs[i, 2].set_title(f'Predicción t+1={i+1}')
        axs[i, 2].set_xlabel(xlabel)
        axs[i, 2].set_ylabel(ylabel)
    
    plt.tight_layout()
    plt.show()

# =====================================================
# PREPROCESAMIENTO DE DATOS
# =====================================================

print("=" * 50)
print("PREPROCESAMIENTO DE IMÁGENES")
print("=" * 50)

# Verificar y redimensionar imágenes si es necesario
print("Verificando si necesitas redimensionar imágenes...")
existing_files = len([f for f in os.listdir(d_reducida) if f.endswith('.npy')])
if existing_files == 0:
    processed_count = resize_npy_images(d_original, d_reducida, new_size=(480, 480))
    print(f"✅ Redimensionadas {processed_count} imágenes")
else:
    print(f"✅ Ya existen {existing_files} imágenes redimensionadas en {d_reducida}")

# Cargar imágenes
file_list = sorted([f for f in os.listdir(d_reducida) if f.endswith('.npy')])
print(f"\nCargando {len(file_list)} imágenes desde {d_reducida}...")

images = []
for i, filename in enumerate(file_list):
    img_path = os.path.join(d_reducida, filename)
    img_array = np.load(img_path)
    images.append(img_array)
    
    if (i + 1) % 100 == 0:
        print(f"  Cargadas {i + 1}/{len(file_list)} imágenes...")

print(f"✅ Carga completada: {len(images)} imágenes")

# Normalización
print("\nNormalizando imágenes...")
images = [(img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8) for img in images]
images = [np.clip(img, 0, 1) for img in images]
print("✅ Normalización completada")

# Construir secuencias temporales
print("\nConstruyendo secuencias temporales...")
timesteps = 3

X_base = np.array(images)[..., np.newaxis]
y_base = X_base.copy()

N_seq = X_base.shape[0] - timesteps
X_seq = np.array([X_base[i:i+timesteps] for i in range(N_seq)])
y_seq = y_base[timesteps:]

print(f"✅ Secuencias creadas:")
print(f"   - Forma de X_seq: {X_seq.shape}")
print(f"   - Forma de y_seq: {y_seq.shape}")
print(f"   - Número de secuencias: {N_seq}")

# =====================================================
# MODELO SIMPLIFICADO - SIN FUNCIONES ANIDADAS
# =====================================================

print("\n" + "=" * 50)
print("CONSTRUYENDO MODELO SIMPLIFICADO")
print("=" * 50)

# Parámetros
H, W, C = X_seq.shape[2], X_seq.shape[3], X_seq.shape[4]
semilla = 42
tf.random.set_seed(semilla)

# =====================================================
# CONSTRUCCIÓN CAPA POR CAPA
# =====================================================

# Capa de entrada
inputs = Input(shape=(timesteps, H, W, C), name='input_sequence')

# =====================================================
# BLOQUE 1: Extracción multi-escala inicial
# =====================================================

# Rama 3x3 del Bloque 1
block1_conv3x3 = ConvLSTM2D(
    16, (3, 3), padding='same', 
    return_sequences=True, 
    kernel_regularizer=l2(1e-4),
    name='block1_conv3x3'
)(inputs)
block1_conv3x3_bn = TimeDistributed(BatchNormalization())(block1_conv3x3)
block1_conv3x3_act = Activation('relu')(block1_conv3x3_bn)

# Rama 5x5 del Bloque 1  
block1_conv5x5 = ConvLSTM2D(
    16, (5, 5), padding='same',
    return_sequences=True,
    kernel_regularizer=l2(1e-4),
    name='block1_conv5x5'
)(inputs)
block1_conv5x5_bn = TimeDistributed(BatchNormalization())(block1_conv5x5)
block1_conv5x5_act = Activation('relu')(block1_conv5x5_bn)

# Fusión por SUMA (no concatenación)
block1_merged = Add()([block1_conv3x3_act, block1_conv5x5_act])
block1_output = Activation('relu', name='block1_output')(block1_merged)
block1_dropout = Dropout(0.2)(block1_output)

# =====================================================
# BLOQUE 2: Con conexión residual
# =====================================================

# Guardar para conexión residual
residual_shortcut = block1_dropout

# Rama 3x3 del Bloque 2
block2_conv3x3 = ConvLSTM2D(
    16, (3, 3), padding='same', 
    return_sequences=True, 
    kernel_regularizer=l2(1e-4),
    name='block2_conv3x3'
)(block1_dropout)
block2_conv3x3_bn = TimeDistributed(BatchNormalization())(block2_conv3x3)
block2_conv3x3_act = Activation('relu')(block2_conv3x3_bn)

# Rama 5x5 del Bloque 2
block2_conv5x5 = ConvLSTM2D(
    16, (5, 5), padding='same',
    return_sequences=True,
    kernel_regularizer=l2(1e-4),
    name='block2_conv5x5'
)(block1_dropout)
block2_conv5x5_bn = TimeDistributed(BatchNormalization())(block2_conv5x5)
block2_conv5x5_act = Activation('relu')(block2_conv5x5_bn)

# Fusión por suma
block2_merged = Add()([block2_conv3x3_act, block2_conv5x5_act])

# Conexión RESIDUAL
block2_residual = Add()([block2_merged, residual_shortcut])
block2_activated = Activation('relu', name='block2_output')(block2_residual)
block2_dropout = Dropout(0.3)(block2_activated)

# =====================================================
# BLOQUE 3: Procesamiento profundo
# =====================================================

# Rama 3x3 del Bloque 3
block3_conv3x3 = ConvLSTM2D(
    32, (3, 3), padding='same', 
    return_sequences=True, 
    kernel_regularizer=l2(1e-4),
    name='block3_conv3x3'
)(block2_dropout)
block3_conv3x3_bn = TimeDistributed(BatchNormalization())(block3_conv3x3)
block3_conv3x3_act = Activation('relu')(block3_conv3x3_bn)

# Rama 5x5 del Bloque 3
block3_conv5x5 = ConvLSTM2D(
    32, (5, 5), padding='same',
    return_sequences=True,
    kernel_regularizer=l2(1e-4),
    name='block3_conv5x5'
)(block2_dropout)
block3_conv5x5_bn = TimeDistributed(BatchNormalization())(block3_conv5x5)
block3_conv5x5_act = Activation('relu')(block3_conv5x5_bn)

# Fusión por suma
block3_merged = Add()([block3_conv3x3_act, block3_conv5x5_act])
block3_output = Activation('relu', name='block3_output')(block3_merged)
block3_dropout = Dropout(0.2)(block3_output)

# =====================================================
# CAPA FINAL
# =====================================================

# Última capa ConvLSTM
final_convLSTM = ConvLSTM2D(
    16, (3, 3), padding='same', 
    return_sequences=False,
    kernel_regularizer=l2(1e-4),
    name='final_convLSTM'
)(block3_dropout)

# Capa de salida
outputs = Conv2D(
    1, (1, 1), padding='same', 
    activation='linear',
    name='output'
)(final_convLSTM)

# =====================================================
# CREAR MODELO
# =====================================================

model14 = Model(inputs=inputs, outputs=outputs, name='ConvLSTM_simplificado_expandido')

# Compilar modelo
initial_lr = 1e-3
optimizer = Adam(learning_rate=initial_lr)

model14.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=[RootMeanSquaredError(), MeanAbsolutePercentageError()]
)

print("✅ Modelo construido capa por capa:")
model14.summary()

# =====================================================
# CALLBACKS
# =====================================================

def cosine_decay_schedule(epoch, lr):
    epochs_total = 50
    min_lr = 1e-5
    
    if epoch < 10:
        return lr
    else:
        cosine_decay = 0.5 * (1 + np.cos(np.pi * (epoch - 10) / (epochs_total - 10)))
        new_lr = min_lr + (initial_lr - min_lr) * cosine_decay
        return new_lr

callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6, verbose=1),
    ModelCheckpoint(filepath="modelo14_best.h5", monitor="val_loss", save_best_only=True, mode="min", verbose=1),
    LearningRateScheduler(cosine_decay_schedule, verbose=1)
]

# =====================================================
# DIVISIÓN DE DATOS
# =====================================================

print("\n" + "=" * 50)
print("DIVISIÓN DE DATOS")
print("=" * 50)

porc_validacion = 0.2
split_index = int(len(X_seq) * (1 - porc_validacion))

X_train, X_val = X_seq[:split_index], X_seq[split_index:]
y_train, y_val = y_seq[:split_index], y_seq[split_index:]

print(f"✅ Datos divididos:")
print(f"   - Entrenamiento: {len(X_train)} muestras")
print(f"   - Validación: {len(X_val)} muestras")

# =====================================================
# ENTRENAMIENTO
# =====================================================

print("\n" + "=" * 50)
print("INICIANDO ENTRENAMIENTO")
print("=" * 50)

history14 = model14.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=4,
    callbacks=callbacks,
    verbose=1
)

print("✅ ¡Entrenamiento completado!")

# =====================================================
# RESULTADOS
# =====================================================

print("\n" + "=" * 50)
print("RESULTADOS DEL ENTRENAMIENTO")
print("=" * 50)

final_train_loss = history14.history['loss'][-1]
final_val_loss = history14.history['val_loss'][-1]
best_epoch = np.argmin(history14.history['val_loss']) + 1

print(f"📊 Resultados finales:")
print(f"   - Pérdida final entrenamiento: {final_train_loss:.4f}")
print(f"   - Pérdida final validación: {final_val_loss:.4f}")
print(f"   - Mejor época: {best_epoch}")

# =====================================================
# VISUALIZACIÓN
# =====================================================

print("\nGenerando predicciones...")
y_pred = model14.predict(X_val[:10])

mostrar_resultados(
    X_val[:10, -1],
    y_val[:10], 
    y_pred[:10],
    n=5,
    titulo_general="Modelo 14 - Arquitectura Simplificada",
    xlabel="Píxeles (X)",
    ylabel="Píxeles (Y)"
)

# Gráficas de entrenamiento
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history14.history['loss'], label='Entrenamiento')
plt.plot(history14.history['val_loss'], label='Validación')
plt.title('Pérdida - Modelo 14 (Simplificado)')
plt.xlabel('Época')
plt.ylabel('Pérdida (MSE)')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history14.history['lr'], label='Learning Rate')
plt.title('Learning Rate durante entrenamiento')
plt.xlabel('Época')
plt.ylabel('Learning Rate')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print("\n🎯 ¡Proceso completado! Modelo guardado: modelo14_best.h5")

PREPROCESAMIENTO DE IMÁGENES
Verificando si necesitas redimensionar imágenes...
✅ Ya existen 741 imágenes redimensionadas en /home/fisica/bandas/banda13/ene_01_reducida_mes

Cargando 741 imágenes desde /home/fisica/bandas/banda13/ene_01_reducida_mes...
  Cargadas 100/741 imágenes...
  Cargadas 200/741 imágenes...
  Cargadas 300/741 imágenes...
  Cargadas 400/741 imágenes...
  Cargadas 500/741 imágenes...
  Cargadas 600/741 imágenes...
  Cargadas 700/741 imágenes...
✅ Carga completada: 741 imágenes

Normalizando imágenes...
✅ Normalización completada

Construyendo secuencias temporales...
✅ Secuencias creadas:
   - Forma de X_seq: (738, 3, 480, 480, 1)
   - Forma de y_seq: (738, 480, 480, 1)
   - Número de secuencias: 738

CONSTRUYENDO MODELO SIMPLIFICADO
✅ Modelo construido capa por capa:


2025-10-21 07:30:36.524690: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "ConvLSTM_simplificado_expandido"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 3, 480,    │          0 │ -                 │
│ (InputLayer)        │ 480, 1)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv3x3      │ (None, 3, 480,    │      9,856 │ input_sequence[0… │
│ (ConvLSTM2D)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_conv5x5      │ (None, 3, 480,    │     27,264 │ input_sequence[0… │
│ (ConvLSTM2D)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 3, 480,    │         64 │ block1_conv3x3[0… │
│ (TimeDistributed)   │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_1  │ (None, 3, 480,    │         64 │ block1_conv5x5[0… │
│ (TimeDistributed)   │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 3, 480,    │          0 │ time_distributed… │
│ (Activation)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 3, 480,    │          0 │ time_distributed… │
│ (Activation)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 3, 480,    │          0 │ activation[0][0], │
│                     │ 480, 16)          │            │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_output       │ (None, 3, 480,    │          0 │ add[0][0]         │
│ (Activation)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 3, 480,    │          0 │ block1_output[0]… │
│                     │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv3x3      │ (None, 3, 480,    │     18,496 │ dropout[0][0]     │
│ (ConvLSTM2D)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_conv5x5      │ (None, 3, 480,    │     51,264 │ dropout[0][0]     │
│ (ConvLSTM2D)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_2  │ (None, 3, 480,    │         64 │ block2_conv3x3[0… │
│ (TimeDistributed)   │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed_3  │ (None, 3, 480,    │         64 │ block2_conv5x5[0… │
│ (TimeDistributed)   │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 3, 480,    │          0 │ time_distributed… │
│ (Activation)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 3, 480,    │          0 │ time_distributed… │
│ (Activation)        │ 480, 16)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 3, 480,    │          0 │ activation_2[0][

 Total params: 344,273 (1.31 MB)

...


DIVISIÓN DE DATOS
✅ Datos divididos:
   - Entrenamiento: 590 muestras
   - Validación: 148 muestras

INICIANDO ENTRENAMIENTO


2025-10-21 07:30:37.118544: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 1631232000 exceeds 10% of free system memory.
2025-10-21 07:30:38.617653: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 543744000 exceeds 10% of free system memory.



Epoch 1: LearningRateScheduler setting learning rate to 0.0010000000474974513.
Epoch 1/50


2025-10-21 07:31:05.094059: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 353894400 exceeds 10% of free system memory.
2025-10-21 07:31:07.656337: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 353894400 exceeds 10% of free system memory.
2025-10-21 07:31:09.615366: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 353894400 exceeds 10% of free system memory.


  1/148 ━━━━━━━━━━━━━━━━━━━━ 7:31:59 184s/step - loss: 0.9272 - mean_absolute_percentage_error: 1577.9529 - root_mean_squared_error: 0.9548

El presente notebook muestra los resultados de la implementación de un model141414o de redes neruonales convolucionales donde se emplean filtros de 3x3 y 5x5 en cada capa y se concatenan las salida para la entrada a la capa siguiente

In [7]:
import os
import numpy as np
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, ConvLSTM2D, Conv2D, TimeDistributed,
    Add, Activation, BatchNormalization, Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, LearningRateScheduler, ReduceLROnPlateau
from tensorflow.keras.metrics import RootMeanSquaredError, MeanAbsolutePercentageError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt


In [8]:
# =====================================================
# CONFIGURACIÓN DE RUTAS
# =====================================================

# Rutas de tus imágenes - AJUSTA ESTAS RUTAS SEGÚN TU SISTEMA
d_original = '/home/fisica/bandas/banda13/ene_01_mes'  # Directorio con imágenes originales
d_reducida = '/home/fisica/bandas/banda13/ene_01_reducida_mes'  # Directorio para imágenes redimensionadas

# Crear directorio reducido si no existe
if not os.path.exists(d_reducida):
    os.makedirs(d_reducida)

In [9]:
# =====================================================
# FUNCIONES AUXILIARES
# =====================================================

def resize_npy_images(d_original, d_reducida, new_size=(480, 480)):
    """Redimensiona imágenes .npy al tamaño especificado"""
    print(f"Redimensionando imágenes en {d_original}...")
    
    if not os.path.exists(d_reducida):
        os.makedirs(d_reducida)
    
    processed_count = 0
    for filename in os.listdir(d_original):
        if filename.lower().endswith('.npy'):
            img_path = os.path.join(d_original, filename)
            img_array = np.load(img_path)
            
            # Redimensionar usando PIL
            img_resized = np.array(Image.fromarray(img_array).resize(new_size, Image.LANCZOS))
            
            # Guardar imagen redimensionada
            output_path = os.path.join(d_reducida, filename)
            np.save(output_path, img_resized)
            processed_count += 1
    
    print(f"Procesadas {processed_count} imágenes")
    return processed_count

def mostrar_resultados(X, y, y_pred, n=5, cmap='viridis',
                       titulo_general="Comparación de Resultados",
                       xlabel="Eje X (pixeles)", ylabel="Eje Y (pixeles)"):
    """
    Muestra comparaciones entre imágenes de entrada, reales y predichas.
    """
    n = min(n, len(X))
    
    fig, axs = plt.subplots(n, 3, figsize=(12, 3 * n))
    fig.suptitle(titulo_general, fontsize=16, y=1.02)
    
    for i in range(n):
        # Entrada
        axs[i, 0].imshow(X[i, ..., 0], cmap=cmap)
        axs[i, 0].set_title(f'Entrada t={i}')
        axs[i, 0].set_xlabel(xlabel)
        axs[i, 0].set_ylabel(ylabel)
        
        # Real
        axs[i, 1].imshow(y[i, ..., 0], cmap=cmap)
        axs[i, 1].set_title(f'Real t+1={i+1}')
        axs[i, 1].set_xlabel(xlabel)
        axs[i, 1].set_ylabel(ylabel)
        
        # Predicción
        axs[i, 2].imshow(y_pred[i, ..., 0], cmap=cmap)
        axs[i, 2].set_title(f'Predicción t+1={i+1}')
        axs[i, 2].set_xlabel(xlabel)
        axs[i, 2].set_ylabel(ylabel)
    
    plt.tight_layout()
    plt.show()


In [10]:
# =====================================================
# PREPROCESAMIENTO DE DATOS
# =====================================================

print("=" * 50)
print("PREPROCESAMIENTO DE IMÁGENES")
print("=" * 50)

# Redimensionar imágenes (solo si es necesario)
print("Verificando si necesitas redimensionar imágenes...")
existing_files = len([f for f in os.listdir(d_reducida) if f.endswith('.npy')])
if existing_files == 0:
    processed_count = resize_npy_images(d_original, d_reducida, new_size=(480, 480))
    print(f"✅ Redimensionadas {processed_count} imágenes")
else:
    print(f"✅ Ya existen {existing_files} imágenes redimensionadas en {d_reducida}")

# Cargar las imágenes redimensionadas
file_list = sorted([f for f in os.listdir(d_reducida) if f.endswith('.npy')])
print(f"\nCargando {len(file_list)} imágenes desde {d_reducida}...")

images = []
for i, filename in enumerate(file_list):
    img_path = os.path.join(d_reducida, filename)
    img_array = np.load(img_path)
    images.append(img_array)
    
    if (i + 1) % 100 == 0:  # Mostrar progreso cada 100 imágenes
        print(f"  Cargadas {i + 1}/{len(file_list)} imágenes...")

print(f"✅ Carga completada: {len(images)} imágenes")

# Normalización
print("\nNormalizando imágenes...")
images = [(img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8) for img in images]
images = [np.clip(img, 0, 1) for img in images]
print("✅ Normalización completada")

# Construir secuencias temporales
print("\nConstruyendo secuencias temporales...")
timesteps = 3

X_base = np.array(images)[..., np.newaxis]  # (num_imgs, H, W, C)
y_base = X_base.copy()

N_seq = X_base.shape[0] - timesteps
X_seq = np.array([X_base[i:i+timesteps] for i in range(N_seq)])  # (N_seq, timesteps, H, W, C)
y_seq = y_base[timesteps:]  # (N_seq, H, W, C)

print(f"✅ Secuencias creadas:")
print(f"   - Forma de X_seq: {X_seq.shape}")  # (N_seq, timesteps, H, W, C)
print(f"   - Forma de y_seq: {y_seq.shape}")  # (N_seq, H, W, C)
print(f"   - Número de secuencias: {N_seq}")


PREPROCESAMIENTO DE IMÁGENES
Verificando si necesitas redimensionar imágenes...
✅ Ya existen 741 imágenes redimensionadas en /home/fisica/bandas/banda13/ene_01_reducida_mes

Cargando 741 imágenes desde /home/fisica/bandas/banda13/ene_01_reducida_mes...
  Cargadas 100/741 imágenes...
  Cargadas 200/741 imágenes...
  Cargadas 300/741 imágenes...
  Cargadas 400/741 imágenes...
  Cargadas 500/741 imágenes...
  Cargadas 600/741 imágenes...
  Cargadas 700/741 imágenes...
✅ Carga completada: 741 imágenes

Normalizando imágenes...
✅ Normalización completada

Construyendo secuencias temporales...
✅ Secuencias creadas:
   - Forma de X_seq: (738, 3, 480, 480, 1)
   - Forma de y_seq: (738, 480, 480, 1)
   - Número de secuencias: 738


In [11]:
# =====================================================
# MODELO SIMPLIFICADO
# =====================================================

def build_simplified_convLSTM(timesteps, H, W, C):
    """
    Versión simplificada del modelo ConvLSTM multiescala
    """
    
    inputs = Input(shape=(timesteps, H, W, C), name='input_sequence')
    
    def multi_scale_block(x, filters, block_name):
        """Bloque multi-escala con fusión por suma"""
        # Rama 3x3
        branch_3x3 = ConvLSTM2D(
            filters, (3, 3), padding='same', 
            return_sequences=True, 
            kernel_regularizer=l2(1e-4),
            name=f'{block_name}_conv3x3'
        )(x)
        branch_3x3 = TimeDistributed(BatchNormalization())(branch_3x3)
        branch_3x3 = Activation('relu')(branch_3x3)
        
        # Rama 5x5  
        branch_5x5 = ConvLSTM2D(
            filters, (5, 5), padding='same',
            return_sequences=True,
            kernel_regularizer=l2(1e-4),
            name=f'{block_name}_conv5x5'
        )(x)
        branch_5x5 = TimeDistributed(BatchNormalization())(branch_5x5)
        branch_5x5 = Activation('relu')(branch_5x5)
        
        # Fusión por suma (en lugar de concatenación)
        merged = Add()([branch_3x3, branch_5x5])
        merged = Activation('relu', name=f'{block_name}_merged')(merged)
        merged = Dropout(0.2)(merged)
        
        return merged
    
    # Arquitectura principal
    x = multi_scale_block(inputs, 16, 'block1')
    
    # Bloque con conexión residual
    shortcut = x
    x = multi_scale_block(x, 16, 'block2')
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    x = Dropout(0.3)(x)
    
    # Bloque final
    x = multi_scale_block(x, 32, 'block3')
    
    # Capa de salida
    x = ConvLSTM2D(
        16, (3, 3), padding='same', 
        return_sequences=False,
        kernel_regularizer=l2(1e-4),
        name='final_convLSTM'
    )(x)
    
    outputs = Conv2D(
        1, (1, 1), padding='same', 
        activation='linear',
        name='output'
    )(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='ConvLSTM_simplified')
    return model

In [ ]:
# =====================================================
# CALLBACKS Y CONFIGURACIÓN DE ENTRENAMIENTO
# =====================================================

def cosine_decay_schedule(epoch, lr):
    """Learning rate schedule mejorado"""
    epochs_total = 50
    min_lr = 1e-5
    
    if epoch < 10:
        return lr
    else:
        cosine_decay = 0.5 * (1 + np.cos(np.pi * (epoch - 10) / (epochs_total - 10)))
        new_lr = min_lr + (initial_lr - min_lr) * cosine_decay
        return new_lr

# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=8,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath="modelo14_simplificado.h5",
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    LearningRateScheduler(cosine_decay_schedule, verbose=1)
]


In [14]:
# =====================================================
# DIVISIÓN DE DATOS
# =====================================================

print("\n" + "=" * 50)
print("DIVISIÓN DE DATOS")
print("=" * 50)

porc_validacion = 0.2
split_index = int(len(X_seq) * (1 - porc_validacion))

X_train, X_val = X_seq[:split_index], X_seq[split_index:]
y_train, y_val = y_seq[:split_index], y_seq[split_index:]

print(f"✅ Datos divididos:")
print(f"   - Entrenamiento: {len(X_train)} muestras")
print(f"   - Validación: {len(X_val)} muestras")
print(f"   - Proporción: {len(X_train)/len(X_seq)*100:.1f}% entrenamiento, {len(X_val)/len(X_seq)*100:.1f}% validación")



DIVISIÓN DE DATOS
✅ Datos divididos:
   - Entrenamiento: 590 muestras
   - Validación: 148 muestras
   - Proporción: 79.9% entrenamiento, 20.1% validación


In [ ]:
# =====================================================
# ENTRENAMIENTO
# =====================================================

print("\n" + "=" * 50)
print("INICIANDO ENTRENAMIENTO")
print("=" * 50)

print(f"Parámetros de entrenamiento:")
print(f"   - Épocas: 50")
print(f"   - Batch size: 4")
print(f"   - Learning rate inicial: {initial_lr}")
print(f"   - Tamaño de entrada: {X_train.shape[1:]}")
print(f"   - Directorio de trabajo: {os.getcwd()}")

history = model14.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=4,
    callbacks=callbacks,
    verbose=1
)

print("✅ ¡Entrenamiento completado!")



INICIANDO ENTRENAMIENTO
Parámetros de entrenamiento:
   - Épocas: 50
   - Batch size: 4
   - Learning rate inicial: 0.001
   - Tamaño de entrada: (3, 480, 480, 1)
   - Directorio de trabajo: /home/fisica/monografia_esp_cd/Modelos


2025-10-20 20:30:01.495608: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 1631232000 exceeds 10% of free system memory.



Epoch 1: LearningRateScheduler setting learning rate to 0.0010000000474974513.
Epoch 1/50


2025-10-20 20:30:03.690311: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 543744000 exceeds 10% of free system memory.
2025-10-20 20:30:31.705597: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 353894400 exceeds 10% of free system memory.
2025-10-20 20:30:33.817082: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 353894400 exceeds 10% of free system memory.
2025-10-20 20:30:37.320766: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 353894400 exceeds 10% of free system memory.


  1/148 ━━━━━━━━━━━━━━━━━━━━ 7:34:56 186s/step - loss: 1.0849 - mean_absolute_percentage_error: 1606.8899 - root_mean_squared_error: 1.0341